# Lesson 38: Visualizing and Understanding CNNs

A trained CNN is a black box in the sense that its millions of weights don't have obvious individual meanings. But *where in the input image* a prediction comes from is answerable, and answering it is often what separates "the model got the right answer" from "the model got the right answer for the right reason." This lesson builds two visualization tools from scratch — **saliency maps** (<a href="../references.html#simonyan-2013-saliency">Simonyan et al., 2013</a>) and **Grad-CAM** (<a href="../references.html#selvaraju-2017">Selvaraju et al., 2017</a><span class="landmark-paper">&#9733;</span>) — on real photos from CIFAR-10 (<a href="../references.html#krizhevsky-2009-cifar">Krizhevsky, 2009</a>), then uses them to catch a model that's cheating.

In [ ]:
import pickle
import tarfile
import urllib.request
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

## Setup: a cat-vs-automobile CNN, with feature maps exposed

A CIFAR-10 binary task: cat vs. automobile, 300 training images per class. The architecture is Lesson 33's CNN pattern, except `forward` now also returns the last convolutional layer's feature map (before global pooling), so both visualization methods below can get at it.

In [ ]:
CIFAR_URL = 'https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz'
CACHE_ROOT = Path.home() / '.cache' / 'cvintro'
CACHE_DIR = CACHE_ROOT / 'cifar-10-batches-py'

def ensure_cifar10():
    if CACHE_DIR.exists():
        return
    CACHE_ROOT.mkdir(parents=True, exist_ok=True)
    archive_path = CACHE_ROOT / 'cifar-10-python.tar.gz'
    if not archive_path.exists():
        print('Downloading CIFAR-10 (~163 MB, one-time, cached under ~/.cache/cvintro)...')
        urllib.request.urlretrieve(CIFAR_URL, archive_path)
    print('Extracting...')
    with tarfile.open(archive_path) as tar:
        tar.extractall(CACHE_ROOT)

def load_cifar_batch(path):
    with open(path, 'rb') as f:
        d = pickle.load(f, encoding='bytes')
    imgs = d[b'data'].reshape(-1, 3, 32, 32).transpose(0, 2, 3, 1).astype(np.float32) / 255.0
    labels = np.array(d[b'labels'], dtype=np.int64)
    return imgs, labels

ensure_cifar10()

CIFAR_LABELS = ['airplane', 'automobile', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
CID = {name: CIFAR_LABELS.index(name) for name in CIFAR_LABELS}

train_imgs, train_labels = [], []
for i in range(1, 6):
    imgs, labels = load_cifar_batch(CACHE_DIR / f'data_batch_{i}')
    train_imgs.append(imgs); train_labels.append(labels)
train_imgs, train_labels = np.concatenate(train_imgs), np.concatenate(train_labels)
test_imgs, test_labels = load_cifar_batch(CACHE_DIR / 'test_batch')

def take(imgs, labels, name, n, rng_local):
    idx = np.where(labels == CID[name])[0]
    idx = rng_local.permutation(idx)[:n]
    return imgs[idx].copy()

rng = np.random.default_rng(4)
X_cat_train = take(train_imgs, train_labels, 'cat', 300, rng)
X_auto_train = take(train_imgs, train_labels, 'automobile', 300, rng)
X_cat_test = take(test_imgs, test_labels, 'cat', 100, rng)
X_auto_test = take(test_imgs, test_labels, 'automobile', 100, rng)

X_train = np.concatenate([X_cat_train, X_auto_train])
y_train = np.array([0.0] * 300 + [1.0] * 300, dtype=np.float32)
X_test = np.concatenate([X_cat_test, X_auto_test])
y_test = np.array([0.0] * 100 + [1.0] * 100, dtype=np.float32)

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 8, 5, padding=2)
        self.conv2 = nn.Conv2d(8, 16, 5, padding=2)
        self.gpool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Linear(16, 1)

    def forward(self, x):
        f1 = F.relu(self.conv1(x))
        f2 = F.relu(self.conv2(f1))  # last conv feature map, full 32x32 resolution
        feat = self.gpool(f2).flatten(1)
        return self.fc(feat).squeeze(-1), f2

torch.manual_seed(0)
model = CNN()
opt = torch.optim.Adam(model.parameters(), lr=0.001)
Xt = torch.tensor(X_train).permute(0, 3, 1, 2); yt = torch.tensor(y_train)
for _ in range(300):
    opt.zero_grad()
    out, _ = model(Xt)
    loss = F.binary_cross_entropy_with_logits(out, yt)
    loss.backward()
    opt.step()

with torch.no_grad():
    out, _ = model(torch.tensor(X_test).permute(0, 3, 1, 2))
    acc = ((out > 0).float() == torch.tensor(y_test)).float().mean().item()
print(f'test accuracy: {acc:.1%}')

## Saliency maps

The idea (<a href="../references.html#simonyan-2013-saliency">Simonyan et al., 2013</a>): take the gradient of the predicted class *score* with respect to every input pixel. A pixel with a large-magnitude gradient is one where a small change would most change the prediction — i.e., a pixel the network is "looking at."

In [ ]:
def saliency_map(model, img_hw3):
    x = torch.tensor(img_hw3).permute(2, 0, 1).unsqueeze(0)
    x.requires_grad_(True)
    score, feat = model(x)
    score.backward()
    return x.grad[0].abs().amax(dim=0).numpy(), feat  # max abs gradient across the 3 color channels

idx = 3
saliency, _ = saliency_map(model, X_test[idx])

fig, axes = plt.subplots(1, 2, figsize=(7, 3.2))
axes[0].imshow(X_test[idx])
axes[0].set_title('input image'); axes[0].axis('off')
axes[1].imshow(saliency, cmap='hot')
axes[1].set_title('saliency map')
axes[1].axis('off')
plt.show()

## Grad-CAM

Raw saliency maps are pixel-level and tend to be noisy. **Grad-CAM** (<a href="../references.html#selvaraju-2017">Selvaraju et al., 2017</a>) instead works on the last convolutional layer's feature maps, which are lower-resolution but far more semantically meaningful: 

1. Take the gradient of the class score with respect to each channel of the last conv feature map.
2. Average each channel's gradient spatially to get one importance weight per channel.
3. Form a weighted sum of the feature-map channels using those weights, then apply ReLU (only positive evidence for the class matters).

The result is a coarse heatmap, the same spatial size as the last conv layer, that can be upsampled back to the input resolution.

In [ ]:
def grad_cam(model, img_hw3, out_size=32):
    x = torch.tensor(img_hw3).permute(2, 0, 1).unsqueeze(0)
    x.requires_grad_(True)
    score, feat = model(x)
    feat.retain_grad()
    score.backward()
    weights = feat.grad[0].mean(dim=(1, 2))  # (channels,) importance per channel
    cam = F.relu((weights[:, None, None] * feat[0]).sum(dim=0))
    cam_up = F.interpolate(cam[None, None], size=(out_size, out_size), mode='bilinear', align_corners=False)
    return cam_up[0, 0].detach().numpy()

cam = grad_cam(model, X_test[idx])

fig, axes = plt.subplots(1, 3, figsize=(10, 3.2))
axes[0].imshow(X_test[idx])
axes[0].set_title('input image'); axes[0].axis('off')
axes[1].imshow(saliency, cmap='hot')
axes[1].set_title('saliency map'); axes[1].axis('off')
axes[2].imshow(cam, cmap='hot')
axes[2].set_title('Grad-CAM'); axes[2].axis('off')
plt.show()

## Do these maps actually track what the model uses?

A pretty overlay on one photo isn't evidence. But real photos don't come with a ground-truth "the object is exactly here" label the way Lesson 33-37's synthetic shapes did, so a distance-to-true-center check isn't available here. Instead, run the honest version of the test these tools are actually *for*: deliberately give the model a shortcut, and check whether the maps correctly catch it.

Add a small, unmistakable 5x5 white marker to the top-left corner of every **automobile** training image only (never on cat images) — a stand-in for a real-world confound, like a watermark, a lab-specific artifact, or a capture-device quirk that happens to correlate with one class. Train a second model, `model_shortcut`, on this corrupted dataset, and compare it against the original `model` above (never exposed to any marker).

In [ ]:
def add_marker(imgs):
    out = imgs.copy()
    out[:, 0:5, 0:5, :] = 1.0  # a stark 5x5 white square, top-left corner
    return out

X_train_shortcut = np.concatenate([X_cat_train, add_marker(X_auto_train)])
X_auto_test_marked = add_marker(X_auto_test)
X_test_marked = np.concatenate([X_cat_test, X_auto_test_marked])  # marker present at test time too

torch.manual_seed(0)
model_shortcut = CNN()
opt2 = torch.optim.Adam(model_shortcut.parameters(), lr=0.001)
Xt_shortcut = torch.tensor(X_train_shortcut).permute(0, 3, 1, 2)
for _ in range(300):
    opt2.zero_grad()
    out, _ = model_shortcut(Xt_shortcut)
    loss = F.binary_cross_entropy_with_logits(out, torch.tensor(y_train))
    loss.backward()
    opt2.step()

def acc_of(m, X, y):
    with torch.no_grad():
        out, _ = m(torch.tensor(X).permute(0, 3, 1, 2))
        return ((out > 0).float() == torch.tensor(y)).float().mean().item()

print(f'{"":>16} {"clean test":>12} {"marked test":>13}')
print(f'{"model":>16} {acc_of(model, X_test, y_test):>11.1%} {acc_of(model, X_test_marked, y_test):>13.1%}')
print(f'{"model_shortcut":>16} {acc_of(model_shortcut, X_test, y_test):>11.1%} {acc_of(model_shortcut, X_test_marked, y_test):>13.1%}')

`model`'s accuracy barely moves whether the marker is present or not — it never learned to use it, so it has nothing to lose when it's absent. `model_shortcut` looks *better* than `model` when the marker is present, and collapses when it's removed: it learned "white corner square = automobile" instead of "automobile = automobile," a shortcut invisible to a single accuracy number computed only on marked data.

Now use Grad-CAM to check whether the visualization tool actually catches this. Since the marker's location is known exactly (rows 0-4, columns 0-4), the "distance to true center" check from a controlled synthetic dataset becomes: what fraction of test images does each model's Grad-CAM peak land inside that exact 5x5 region?

In [ ]:
def frac_peak_in_marker(m, imgs):
    count = 0
    for img in imgs:
        cam_i = grad_cam(m, img)
        peak = np.unravel_index(cam_i.argmax(), cam_i.shape)  # (row, col)
        if peak[0] < 5 and peak[1] < 5:
            count += 1
    return count / len(imgs)

frac_clean_on_marked = frac_peak_in_marker(model, X_auto_test_marked)
frac_shortcut_on_marked = frac_peak_in_marker(model_shortcut, X_auto_test_marked)
frac_shortcut_on_clean = frac_peak_in_marker(model_shortcut, X_auto_test)

print(f'{"model":>16} {"test images":>14} {"Grad-CAM peak in marker region":>32}')
print(f'{"model":>16} {"marked":>14} {frac_clean_on_marked:>31.1%}')
print(f'{"model_shortcut":>16} {"marked":>14} {frac_shortcut_on_marked:>31.1%}')
print(f'{"model_shortcut":>16} {"clean (no marker!)":>14} {frac_shortcut_on_clean:>31.1%}')

`model` (never trained on the marker) sometimes lands in that corner purely because a stark white square is such a strong local gradient signal that it draws some attention from any model, marker-dependent or not — a real limitation worth remembering when reading a single overlay. `model_shortcut` is a different story entirely: its Grad-CAM peak lands in the marker's exact 5x5 region *every time* — even on the clean test images where the marker was never added at all. That's the most convincing evidence a visualization tool can offer: the explanation doesn't just correlate with the marker's presence, it reveals that the model's internal machinery is anchored to that specific spot in the image, regardless of what's actually there. Exactly the failure mode described at the top of this lesson — a network getting the right answer for the wrong reason — caught red-handed by the same tool built from scratch above.

The practical lesson: accuracy alone (Lesson 35) can't distinguish a model that learned the real signal from one that learned a shortcut correlated with it in the training data. Saliency maps and Grad-CAM can — but only if you know to check, and only if you interpret a single bright overlay with some skepticism about what else in the image might be drawing gradient attention.

### Exercise

1. Grad-CAM here uses the *last* conv layer. Modify `grad_cam` to instead use the intermediate feature map after `conv1` (before `conv2`). Does the resulting heatmap get sharper (closer to pixel-perfect, like the saliency map) or coarser, and why would an earlier layer behave that way?
2. Shrink the marker from 5x5 to 2x2, or dim it from pure white (`1.0`) to a faint gray (`0.6`). Does `model_shortcut` still learn to rely on it as strongly (check the clean-vs-marked accuracy gap), and does Grad-CAM still catch it as reliably?
3. The saliency-map gradient in this lesson is taken with respect to the raw logit (`score`), not the sigmoid probability. Try computing it with respect to `torch.sigmoid(score)` instead — does the resulting map look meaningfully different, and can you explain why using the chain rule?